# Data Processing

In [ ]:
import json
import os

# Load global paths from JSON
try:
    with open("processing_paths.json", 'r') as f:
        paths = json.load(f)
    print("✅ Paths loaded successfully from processing_paths.json")
except FileNotFoundError:
    print("❌ Error: processing_paths.json not found. Please ensure it exists in the same directory.")
    paths = {}

base_directory = os.path.abspath(paths.get('base_directory', ''))

def get_abs_path(key, default=''):
    val = paths.get(key, default)
    if val and not os.path.isabs(val):
        return os.path.abspath(os.path.join(base_directory, val))
    return os.path.abspath(val) if val else ''

SUMEDI_JSON_WCS_PATH = get_abs_path('sumedi_json_wcs_path')
MOSH_C3D_TEMPLATE = get_abs_path('mosh_c3d_template')
CONSTRUCTED_C3D_PATH = get_abs_path('constructed_c3d_path')
PRIMED_C3D_PATH = get_abs_path('primed_c3d_path')
SOMA_SUPPORT_BASE = get_abs_path('soma_support_base')
OUTPUT_ROOT_DIR = get_abs_path('output_root_dir')
NPZ_ANIMATIONS_PATH = get_abs_path('npz_animations_path')
MARKERSET_YAML = paths.get('markerset_yaml', 'superset') 

## 1. JSON to C3D Conversion

In [ ]:
import numpy as np
import ezc3d
import glob
import shutil

def apply_x_rotation_plus_90(points_data):
    rotated_data = points_data.copy()
    old_x, old_y, old_z = points_data[0, :, :], points_data[1, :, :], points_data[2, :, :]
    rotated_data[0, :, :] = -old_x
    rotated_data[1, :, :] = -old_z
    rotated_data[2, :, :] = old_y
    return rotated_data

def convert_json_to_c3d(json_file, template_path, output_path):
    try:
        with open(json_file, 'r') as f: json_data = json.load(f)
    except Exception as e: return
    if not json_data: return
    
    marker_labels = json_data[0].get('point_ids', [])
    n_markers, n_frames = len(marker_labels), len(json_data)
    
    c3d = ezc3d.c3d(template_path)
    if 'meta_points' in c3d['data']: del c3d['data']['meta_points']
    
    c3d['parameters']['POINT']['LABELS']['value'] = marker_labels
    c3d['parameters']['POINT']['USED']['value'] = [n_markers]
    c3d['parameters']['POINT']['FRAMES']['value'] = [n_frames]
    c3d['data']['analogs'] = np.zeros((1, 0, n_frames))
    
    points_data = np.zeros((4, n_markers, n_frames))
    for i, frame in enumerate(json_data):
        xyz = frame.get('xyz', [])
        if len(xyz) != n_markers: xyz = xyz[:n_markers] + [[np.nan]*3]*(n_markers-len(xyz))
        coords = np.array(xyz).T
        valid_mask = ~np.isnan(coords).any(axis=0)
        points_data[0:3, valid_mask, i] = coords[:, valid_mask]
        points_data[3, valid_mask, i], points_data[3, ~valid_mask, i] = 0.0, -1.0
    
    points_data = apply_x_rotation_plus_90(points_data)
    c3d['data']['points'] = points_data
    c3d['header']['points']['first_frame'], c3d['header']['points']['last_frame'] = 0, n_frames - 1
    c3d.write(output_path)

if not os.path.exists(CONSTRUCTED_C3D_PATH): os.makedirs(CONSTRUCTED_C3D_PATH)
subject_folders = [f.path for f in os.scandir(SUMEDI_JSON_WCS_PATH) if f.is_dir()]
for subject_folder in subject_folders:
    subject_name = os.path.basename(subject_folder)
    output_subject_folder = os.path.join(CONSTRUCTED_C3D_PATH, subject_name)
    os.makedirs(output_subject_folder, exist_ok=True)
    for f in glob.glob(os.path.join(subject_folder, "*.[jJ][sS][oO][nN]")):
        convert_json_to_c3d(f, MOSH_C3D_TEMPLATE, os.path.join(output_subject_folder, os.path.splitext(os.path.basename(f))[0] + ".c3d"))
    if os.path.exists(os.path.join(subject_folder, 'settings.json')):
        shutil.copy(os.path.join(subject_folder, 'settings.json'), output_subject_folder)
print("--- JSON to C3D Complete ---")

## 2. SOMA label_priming

In [ ]:
from omegaconf import OmegaConf
import os.path as osp
from soma.tools.run_soma_multiple import run_soma_on_multiple_settings

def resolve_mocap_subject(path):
    return os.path.basename(os.path.dirname(path))
if not OmegaConf.has_resolver("resolve_mocap_subject"):
    OmegaConf.register_new_resolver("resolve_mocap_subject", resolve_mocap_subject, replace=True)

target_dataset_name = osp.relpath(CONSTRUCTED_C3D_PATH, base_directory)
os.makedirs(PRIMED_C3D_PATH, exist_ok=True)

run_soma_on_multiple_settings(
    soma_expr_ids=['V48_02_SuperSet'], 
    soma_data_ids=['OC_05_G_03_real_000_synt_100'],
    soma_mocap_target_ds_names=[target_dataset_name],
    soma_cfg={
        'soma.batch_size': 256,
        'dirs.support_base_dir': SOMA_SUPPORT_BASE,
        'dirs.work_dir': PRIMED_C3D_PATH,
        'dirs.mocap_out_fname': osp.join(PRIMED_C3D_PATH, '${mocap.subject_name}', '${mocap.basename}.pkl'),
        'mocap.unit': 'mm', 
        'save_c3d': True,
        'keep_nan_points': True,
        'remove_zero_trajectories': True
    },
    parallel_cfg={'max_num_jobs': 4, 'randomly_run_jobs': False},
    run_tasks=['soma'],
    mocap_base_dir=base_directory,
    soma_work_base_dir=SOMA_SUPPORT_BASE,
    mocap_ext='.c3d'
)

# Collection Logic
soma_eval_dir = osp.join(PRIMED_C3D_PATH, 'training_experiments', 'V48_02_SuperSet', 'OC_05_G_03_real_000_synt_100', 
                         'evaluations', 'soma_labeled_mocap_tracklet', target_dataset_name)
if osp.exists(soma_eval_dir):
    for root, _, files in os.walk(soma_eval_dir):
        for file in files:
            if file.endswith('.c3d'):
                dest = osp.join(PRIMED_C3D_PATH, osp.relpath(root, soma_eval_dir))
                os.makedirs(dest, exist_ok=True)
                shutil.copy2(osp.join(root, file), osp.join(dest, file))

for s_folder in [d for d in os.listdir(CONSTRUCTED_C3D_PATH) if d.startswith('S')]:
    src_settings = osp.join(CONSTRUCTED_C3D_PATH, s_folder, 'settings.json')
    if osp.exists(src_settings): shutil.copy2(src_settings, osp.join(PRIMED_C3D_PATH, s_folder, 'settings.json'))
print("--- SOMA Priming Complete ---")

## 3. Solve for Body Shape and Motion

In [ ]:
from loguru import logger
from concurrent.futures import ProcessPoolExecutor
from soma.amass.mosh_manual import mosh_manual
from functools import partial

def solve_subject(subject_id, mocap_base, out_root, support_dir, markerset_yaml):
    subject_dir = osp.join(mocap_base, subject_id)
    mocap_fnames = sorted(glob.glob(osp.join(subject_dir, "*.c3d")))
    if not mocap_fnames: return f"Skipped {subject_id}"
    
    gender = 'neutral'
    if osp.exists(osp.join(subject_dir, 'settings.json')):
        with open(osp.join(subject_dir, 'settings.json'), 'r') as f: gender = json.load(f).get('gender', 'neutral')
    
    mosh_manual(mocap_fnames, mosh_cfg={'moshpp.fall_back_gender': gender, 'moshpp.marker_layout': markerset_yaml, 
                'moshpp.stagei_frame_picker.stagei_mocap_fnames': mocap_fnames, 'dirs.work_base_dir': out_root, 'dirs.support_base_dir': support_dir},
                render_cfg={'dirs.work_base_dir': out_root, 'dirs.temp_base_dir': osp.join(out_root, 'tmp', subject_id), 'dirs.support_base_dir': support_dir},
                parallel_cfg={'pool_size': 4, 'max_num_jobs': 1000}, run_tasks=['mosh'])
    return f"✅ {subject_id} Finished"

subject_folders = sorted([d for d in os.listdir(PRIMED_C3D_PATH) if d.startswith('S') and osp.isdir(osp.join(PRIMED_C3D_PATH, d))])
with ProcessPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(partial(solve_subject, mocap_base=PRIMED_C3D_PATH, out_root=OUTPUT_ROOT_DIR, 
                                      support_dir=osp.join(SOMA_SUPPORT_BASE, 'support_files', markerset_yaml=MARKERSET_YAML)), subject_folders))
for res in results: print(res)

## 4. Animate (Convert PKL to NPZ)

In [ ]:
import pickle
from tqdm import tqdm

solve_root = os.path.join(OUTPUT_ROOT_DIR, os.path.basename(PRIMED_C3D_PATH))
os.makedirs(NPZ_ANIMATIONS_PATH, exist_ok=True)

for subject in tqdm(sorted([d for d in os.listdir(solve_root) if d.startswith('S')]), desc="Converting"):
    subj_solve, subj_out = os.path.join(solve_root, subject), os.path.join(NPZ_ANIMATIONS_PATH, subject)
    os.makedirs(subj_out, exist_ok=True)
    gender = 'neutral'
    if osp.exists(osp.join(PRIMED_C3D_PATH, subject, 'settings.json')):
        with open(osp.join(PRIMED_C3D_PATH, subject, 'settings.json'), 'r') as f: gender = json.load(f).get('gender', 'neutral')
    
    for pkl in glob.glob(os.path.join(subj_solve, "*_stageii.pkl")):
        out_npz = os.path.join(subj_out, os.path.basename(pkl).replace('_stageii.pkl', '_rectified.npz'))
        if os.path.exists(out_npz): continue
        with open(pkl, 'rb') as f: data = pickle.load(f, encoding='latin1')
        np.savez(out_npz, gender=gender, surface_model_type='smplx', mocap_frame_rate=np.array(30.0), 
                 mocap_time_length=np.array(float(len(data['trans']))), trans=data['trans'], poses=data['fullpose'], 
                 betas=data['betas'][:100].astype(np.float32), num_betas=np.array(100))
print("✨ All steps complete!")